[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab13_pandas_cleaning/lab13_student.ipynb)

# Lab 13 — pandas II: Cleaning Text Data

### Digital Skills and Business Decisioning Using Python

In Lab 12 you explored the reviews dataset and found problems along the way: an extra index column, missing titles and reviews, a misspelled division name (`Initmates`), and column names with spaces in them. That's normal. Real datasets are always messy, and analysts commonly say that **most of the work in a data project is cleaning**.

Cleaning isn't a mechanical checklist. Every step is a **decision** (drop these rows or fill them in? fix this value or leave it?), and each decision changes the numbers you'll report later. So this lab does two things: it teaches the pandas tools for cleaning, and it asks you to justify each choice.

You'll also bring back the text tools from Labs 08 and 09. pandas can run string methods and regular expressions on a whole column at once, which turns the review text into numeric **features** (length, exclamation marks, a customer's height) that you can analyse in the next labs.

In this lab you'll learn to:

* **rename** columns,
* find and handle **missing values** with `isna()`, `dropna()`, and `fillna()`,
* find and remove **duplicates**,
* fix inconsistent **category values** with `replace()`, and change **types** with `astype()`,
* work with **dates** using `pd.to_datetime()`,
* process text columns with the **`.str` accessor** (including regex), and
* apply **your own functions** to a column with `.apply()` and `.map()`.

### Setup

Load the data as in Lab 12, with `index_col=0` to absorb the extra index column.

In [ ]:
import re
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/JanNikolaidovic/python-text-analytics-lab/main/data/womens_clothing_ecommerce_reviews.csv"

df = pd.read_csv(DATA_URL, index_col=0)
print(df.shape)
df.head()

### Part 1: Tidy Column Names

Column names like `Positive Feedback Count` are readable, but awkward to type, and easy to get wrong (was it `Recommended IND` or `Recommended Ind`?). A common convention is **snake_case**: lowercase, with underscores instead of spaces. `df.columns` can be replaced with a new list of names, and a list comprehension builds that list from the old names:

In [ ]:
df.columns = [name.lower().replace(" ", "_") for name in df.columns]
df.columns

Better. A few names are still longer than they need to be. **`.rename(columns={old: new})`** changes specific columns using a dictionary. Like most pandas methods, it returns a **new** DataFrame, so assign the result back to `df`:

In [ ]:
df = df.rename(columns={
    "recommended_ind": "recommended",
    "division_name": "division",
    "department_name": "department",
})
df.columns

(`class_name` keeps its `_name`: `class` on its own is a reserved word in Python, and although pandas would allow it as a column name, it's asking for confusion.)

> **Most pandas methods don't change the DataFrame in place.** `df.rename(...)` on its own line does nothing lasting: it creates a renamed copy and throws it away. Always assign the result: `df = df.rename(...)`. This is the same idea as `sorted()` vs `.sort()` from Lab 04, except that in pandas the "return a new copy" behaviour is the norm.

---

### Part 2: Missing Values

pandas marks a missing value as `NaN` ("not a number", even in text columns). **`.isna()`** returns `True` wherever a value is missing, and `.sum()` counts them, column by column:

In [ ]:
df.isna().sum()

Counts are easier to judge as percentages:

In [ ]:
(df.isna().mean() * 100).round(2)

Before deciding what to do, **look at the rows** with missing values. Are they otherwise normal?

In [ ]:
df[df["review_text"].isna()].head()

These customers gave a star rating (and often a recommendation) but wrote nothing. That's a real behaviour, not a data error. For a *text* analysis, though, a review without text is no use.

Here's a reasonable decision for each column. The important thing is that each one has a reason:

| Column | Missing | Decision | Reason |
|---|---|---|---|
| `review_text` | ~3.6% | **drop** those rows | This is a text analysis; there's nothing to analyse. |
| `title` | ~16% | **fill** with `""` (empty text) | The review itself is still valuable; a title is optional. |
| `division`, `department`, `class_name` | 14 rows | **drop** those rows | Too few to matter, and we can't guess the category. |

### Dropping Rows: `dropna()`

**`.dropna(subset=[...])`** drops the rows that have a missing value **in the columns you list**. Always check the size before and after, so you know exactly how much data a decision cost you:

In [ ]:
before = len(df)
df = df.dropna(subset=["review_text", "division", "department", "class_name"])
print(f"Dropped {before - len(df)} rows; {len(df)} remain.")

> **Careful with `dropna()` without `subset`.** On its own, `df.dropna()` drops every row with a missing value in **any** column. Here that would also throw away every review without a title, over 3,000 perfectly good reviews:

In [ ]:
print(len(df), "rows now;", len(df.dropna()), "if we had dropped every row with any missing value")

### Filling In: `fillna()`

**`.fillna(value)`** replaces missing values with a value you choose. For the titles, an empty string means "no title" without breaking any text processing later (you can't lowercase a `NaN`, but you can lowercase `""`):

In [ ]:
df["title"] = df["title"].fillna("")

df.isna().sum()

Every column now shows `0`. The assignment `df["title"] = ...` replaces the whole column with the filled version: that's how you change a column in pandas.

### Practice: Missing Values

**Exercise 1: How Much Did We Lose? (easy)**

The original file had 23,486 rows. Print the share of rows that the cleaning so far removed, as a percentage with 1 decimal.

*Expected result:* `3.7%`

In [ ]:
# TODO: print the share of the original 23,486 rows that were removed

In [ ]:
# @solution
removed = 23486 - len(df)
print(f"{removed / 23486:.1%}")

**Exercise 2: Filling a Number (medium)**

The small DataFrame below has missing prices. Fill the missing values in `price` with the **median** of the prices that are present (`.median()` skips missing values automatically), and display the result. In a comment, give one reason why the median can be a safer choice than the mean for filling in missing values.

In [ ]:
products = pd.DataFrame({"item": ["dress", "tank", "coat", "skirt", "jeans"],
                         "price": [148.0, None, 598.0, 88.0, None]})

# TODO: fill the missing prices with the median price, then display products

In [ ]:
# @solution
products = pd.DataFrame({"item": ["dress", "tank", "coat", "skirt", "jeans"],
                         "price": [148.0, None, 598.0, 88.0, None]})

products["price"] = products["price"].fillna(products["price"].median())
products
# The mean is pulled up by extreme values (the 598 coat); the median is not,
# so it's a more "typical" value to fill in.

---

### Part 3: Duplicates

**`.duplicated()`** marks every row that is an exact copy of an earlier row. For text data, the more useful check is usually on **one column**: the same review text appearing twice.

In [ ]:
print(df.duplicated().sum())
print(df.duplicated(subset=["review_text"]).sum())

One complete row is an exact copy of another, and 7 review texts are repeats of an earlier one. Look at them before deciding. `keep=False` marks **every** copy (not just the second one), and sorting by the text puts each pair next to each other:

In [ ]:
dupes = df[df.duplicated(subset=["review_text"], keep=False)]
dupes.sort_values("review_text")[["clothing_id", "age", "rating", "review_text"]]

Within each group, the age and rating match. Some copies have the **same** `clothing_id` (probably a review submitted twice), and others a **different** one (most likely one customer who bought several items and pasted the same review on each). Neither is exactly wrong, but for counting words, the same text twice would count its words twice. Keep the first copy of each:

In [ ]:
df = df.drop_duplicates(subset=["review_text"])
print(len(df))

> **Not every repeated value is a duplicate.** Titles like `Love it!` appear over a hundred times, and that's completely legitimate: many customers write the same short title. Only remove repeats when the repeated thing *should* be unique. A 60-word review written identically by two different people is almost impossible; a 2-word title is not.

---

### Part 4: Fixing Categories and Types

In Lab 12 you spotted the misspelled division `Initmates`. `value_counts()` shows the problem clearly:

In [ ]:
df["division"].value_counts()

**`.replace({old: new})`** swaps specific values in a column:

In [ ]:
df["division"] = df["division"].replace({"Initmates": "Intimates"})
df["division"].value_counts()

Always run `value_counts()` on every category column early in a project. Typos, inconsistent capitals (`Dresses` vs `dresses`), and stray spaces (`"Tops "`) all show up as separate categories, silently splitting your groups.

### Changing Types: `astype()`

`recommended` is stored as `1`/`0`, but it's really a yes/no value. **`.astype(bool)`** converts it: `1` becomes `True` and `0` becomes `False`. That makes the meaning obvious, and filters read more naturally (`df[df["recommended"]]`).

In [ ]:
df["recommended"] = df["recommended"].astype(bool)

print(df["recommended"].dtype)
print(df["recommended"].mean())

Booleans still behave like `1` and `0` in calculations, so `.mean()` is the share of reviews that recommend the product.

### Dates: `pd.to_datetime()`

This dataset has no dates, but most review datasets do (and yours might in the final assignment), so here's the essential tool. Dates in a CSV file are loaded as **text**. **`pd.to_datetime()`** converts them into real dates, and the **`.dt` accessor** then gives you their parts:

In [ ]:
orders = pd.DataFrame({"order_date": ["2024-03-01", "2024-03-15", "2024-11-29", "2025-01-06"],
                       "amount": [148.0, 38.0, 198.0, 98.0]})

orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["year"] = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.month
orders["weekday"] = orders["order_date"].dt.day_name()

print(orders.dtypes)
orders

Once a column is a real date (`datetime64`), you can sort it chronologically, filter by range (`orders[orders["order_date"] >= "2024-06-01"]`), and group by year or month (next lab). As text, `"2024-11-29"` is just a string.

Some datasets store dates as a number of seconds since 1 January 1970 (a **Unix timestamp**). `pd.to_datetime(column, unit="s")` converts those.

### Practice: Categories and Types

**Exercise 3: Normalize Inconsistent Labels (medium)**

The `sizes` Series below has the same sizes written in different ways. Use `.str.strip()` and `.str.upper()` (string methods for a whole column: the next part explains them fully) to make them consistent, then print the `value_counts()`.

*Expected result:* `S` 3, `M` 2, `XS` 1

In [ ]:
sizes = pd.Series(["S", "s ", " M", "m", "XS", "S"])

# TODO: strip spaces and convert to upper case, then print value_counts()

In [ ]:
# @solution
sizes = pd.Series(["S", "s ", " M", "m", "XS", "S"])

clean_sizes = sizes.str.strip().str.upper()
print(clean_sizes.value_counts())

---

### Part 5: The `.str` Accessor, String Methods for a Whole Column

Every string method you know (`.lower()`, `.split()`, `.replace()`, ...) works on a single string. To run one on **every value in a column**, put **`.str`** in between. pandas applies the method to each value and returns a new Series:

In [ ]:
df["title"].str.lower().head()

`.str.len()` gives the number of characters in each value. To count **words**, split each text into a list and take the length of each list: `.str.split().str.len()`.

In [ ]:
df["n_chars"] = df["review_text"].str.len()
df["n_words"] = df["review_text"].str.split().str.len()

df[["review_text", "n_chars", "n_words"]].head()

Two new **features** in two lines, computed for all ~22,600 reviews. In Lab 11 you needed a loop to build a length array for 600 reviews.

### Regex on a Column

Most `.str` methods accept **regular expressions** from Lab 09:

* `.str.contains(pattern)`: `True`/`False` for each value (does it match anywhere?),
* `.str.count(pattern)`: how many matches in each value,
* `.str.replace(pattern, new, regex=True)`: substitute matches, like `re.sub()`,
* `.str.extract(pattern)`: pull out the **groups**, like `match.group(1)`.

Start with `contains` and `count`:

In [ ]:
df["n_exclaims"] = df["review_text"].str.count("!")
df["mentions_petite"] = df["review_text"].str.contains(r"\bpetite\b", case=False)

print(df["n_exclaims"].mean())
print(df["mentions_petite"].mean())

`case=False` makes the match ignore case (like `re.IGNORECASE`). About 1 in 12 reviews mentions petite sizing, and reviews contain about 0.64 exclamation marks on average (roughly two for every three reviews).

`.str.replace()` with `regex=True` cleans text in the whole column at once. Recall from Lab 09 that some reviews contain line breaks (`\r\n`); collapse every run of whitespace into one space:

In [ ]:
print(df["review_text"].str.contains(r"[\r\n]").sum(), "reviews contain line breaks")

df["review_text"] = df["review_text"].str.replace(r"\s+", " ", regex=True).str.strip()

print(df["review_text"].str.contains(r"[\r\n]").sum(), "reviews contain line breaks after cleaning")

> **Always write `regex=True` with `.str.replace()`.** Without it, pandas treats the pattern as plain text and looks for the literal characters `\s+`, finding nothing, with no error to warn you.

### Extracting With Groups

**`.str.extract()`** returns a DataFrame with **one column per group** in the pattern. Here's the height pattern from Lab 09 applied to every review at once:

In [ ]:
height_parts = df["review_text"].str.extract(r"(\d)'\s?(\d{1,2})")
height_parts.dropna().head()

Column `0` is the first group (feet) and column `1` the second (inches). Reviews with no height get `NaN` in both. The extracted parts are **text**, so convert them to numbers with `.astype(float)` before doing arithmetic, which is vectorized just like NumPy:

In [ ]:
feet = height_parts[0].astype(float)
inches = height_parts[1].astype(float)
df["height_cm"] = ((feet * 12 + inches) * 2.54).round()

print(df["height_cm"].notna().sum(), "reviews mention a height")
df["height_cm"].describe()

`.notna()` is the opposite of `.isna()`. Just over 4,000 reviews include a height, with a median of 165 cm. But look at the **min** and **max**: 61 cm and 307 cm are not real heights. The pattern must be matching something else in a few reviews (perhaps a measurement like `2'' x 3''`, or a typo). You'll track those down in the EDA lab. For now, notice that a quick `describe()` on a new feature is the fastest way to catch problems like this.

### Practice: The `.str` Accessor

**Exercise 4: Title Features (easy)**

Add two columns: `title_words`, the number of words in each title (titles can be empty strings now, which have 0 words), and `title_has_exclaim`, `True` if the title contains `!`. Print the mean of each.

In [ ]:
# TODO: add title_words and title_has_exclaim, then print the mean of each

In [ ]:
# @solution
df["title_words"] = df["title"].str.split().str.len()
df["title_has_exclaim"] = df["title"].str.contains("!", regex=False)

print(df["title_words"].mean())
print(df["title_has_exclaim"].mean())

**Exercise 5: Who Mentions Returns? (medium)**

Add a column `mentions_return` that is `True` when the review contains a word starting with `return` (`return`, `returned`, `returning`, `returns`), ignoring case. Use `\breturn\w*` with `.str.contains()`. Then, for each rating from 1 to 5, print the share of reviews that mention a return. Use a `for` loop and `df.loc[df["rating"] == r, "mentions_return"].mean()`.

In [ ]:
# TODO: add mentions_return, then print the share for each rating 1-5

In [ ]:
# @solution
df["mentions_return"] = df["review_text"].str.contains(r"\breturn\w*", case=False)

for r in range(1, 6):
    share = df.loc[df["rating"] == r, "mentions_return"].mean()
    print(f"{r} stars: {share:.1%}")

(`df.loc[condition, "column"]` combines filtering and column selection in one step: the rows where the condition is `True`, and just that column.)

---

### Part 6: Your Own Functions: `.apply()` and `.map()`

Sometimes the cleaning you need is more than one string method: a whole function, like `clean_text()` from Lab 09. **`.apply(function)`** calls your function on **every value** in a column and collects the results into a new Series. You pass the function **without parentheses**, exactly like `key=` in `sorted()` (Lab 08).

In [ ]:
def clean_text(text):
    """Lowercase text, keep only letters, apostrophes and spaces, and collapse whitespace."""
    text = text.lower()
    text = re.sub(r"[^a-z'\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df["clean_text"] = df["review_text"].apply(clean_text)
df[["review_text", "clean_text"]].head()

Every review now has a cleaned version, ready for word counting. `.apply()` works with any function that takes one value and returns one value, so everything you wrote in Labs 06 to 10 can now run on a whole column.

> **Prefer `.str` methods when one exists.** `.apply()` runs your Python function once per row, which is fine for 20,000 reviews but much slower than a built-in vectorized `.str` method. Use `.apply()` when there's no built-in that does what you need.

### Mapping Values With a Dictionary: `.map()`

A very common job is translating each value through a lookup table. **`.map(dictionary)`** replaces every value with the dictionary's value for that key. Group the 5-star ratings into three **sentiment** labels:

In [ ]:
sentiment_labels = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}

df["sentiment"] = df["rating"].map(sentiment_labels)
df["sentiment"].value_counts()

This `sentiment` column will be important in the next lab, where you'll decide what the **dependent variable** of the analysis should be.

`.map()` also accepts a function, like `.apply()`. The difference to remember: with a dictionary, any value that isn't a key becomes `NaN`, so check the result for missing values after mapping.

### Practice: `.apply()` and `.map()`

**Exercise 6: Length Buckets (medium)**

Write a function `length_bucket(n)` that returns `"short"` if `n < 30`, `"medium"` if `n < 80`, and `"long"` otherwise. Apply it to `n_words` to create a column `length_group`, then print its `value_counts()`.

In [ ]:
# TODO: define length_bucket(n), apply it to n_words as length_group, print value_counts()

In [ ]:
# @solution
def length_bucket(n):
    if n < 30:
        return "short"
    elif n < 80:
        return "medium"
    return "long"

df["length_group"] = df["n_words"].apply(length_bucket)
print(df["length_group"].value_counts())

**Exercise 7: Map the Departments (easy)**

Management wants two broad groups: `"Tops"`, `"Jackets"`, and `"Intimate"` count as `"upper"`; `"Bottoms"` counts as `"lower"`; `"Dresses"` and `"Trend"` count as `"full"`. Create a column `body_area` with `.map()` and print its `value_counts()`. Then check that no value became `NaN`.

In [ ]:
# TODO: map department to body_area with a dictionary, print value_counts() and the number of NaN values

In [ ]:
# @solution
area = {"Tops": "upper", "Jackets": "upper", "Intimate": "upper",
        "Bottoms": "lower", "Dresses": "full", "Trend": "full"}

df["body_area"] = df["department"].map(area)
print(df["body_area"].value_counts())
print(df["body_area"].isna().sum())

---

### Part 7: The Clean Dataset

Here's everything the cleaned DataFrame now contains: the original columns (renamed and fixed), plus the features you engineered.

In [ ]:
df.info()

As a last step, save it. `.to_csv()` writes a DataFrame to a CSV file (`index=False` leaves out the index, so you don't create another `Unnamed: 0` column for whoever loads the file next!).

As always in Colab, the file lives in the temporary runtime and disappears when it ends (Lab 07). That's fine here: the next labs rebuild the clean dataset from the original file with a function, so every notebook is **reproducible** from start to finish.

In [ ]:
df.to_csv("reviews_clean.csv", index=False)
print("Saved reviews_clean.csv")

### Practice: A Reusable Cleaning Function

**Exercise 8: `load_reviews()` (medium)**

Every lab from now on starts by loading and cleaning the data, so it's worth packaging the steps into one function. Write `load_reviews(url)` that:

1. loads the CSV with `index_col=0`,
2. renames the columns (snake_case, plus the three renames from Part 1),
3. drops rows missing `review_text` or the category columns, fills missing titles with `""`, and drops duplicate review texts,
4. fixes `Initmates`, converts `recommended` to `bool`, and collapses whitespace in `review_text`,
5. adds `n_words`, `n_exclaims`, and `sentiment`,

and **returns** the DataFrame. Call it and print the shape of the result.

*Expected result:* `(22621, 13)`

In [ ]:
# TODO: define load_reviews(url), call it with DATA_URL, print the shape

In [ ]:
# @solution
def load_reviews(url):
    reviews = pd.read_csv(url, index_col=0)
    reviews.columns = [name.lower().replace(" ", "_") for name in reviews.columns]
    reviews = reviews.rename(columns={"recommended_ind": "recommended",
                                      "division_name": "division",
                                      "department_name": "department"})
    reviews = reviews.dropna(subset=["review_text", "division", "department", "class_name"])
    reviews["title"] = reviews["title"].fillna("")
    reviews = reviews.drop_duplicates(subset=["review_text"])
    reviews["division"] = reviews["division"].replace({"Initmates": "Intimates"})
    reviews["recommended"] = reviews["recommended"].astype(bool)
    reviews["review_text"] = reviews["review_text"].str.replace(r"\s+", " ", regex=True).str.strip()
    reviews["n_words"] = reviews["review_text"].str.split().str.len()
    reviews["n_exclaims"] = reviews["review_text"].str.count("!")
    reviews["sentiment"] = reviews["rating"].map({1: "negative", 2: "negative", 3: "neutral",
                                                  4: "positive", 5: "positive"})
    return reviews


reviews = load_reviews(DATA_URL)
print(reviews.shape)

---

### Recap: Your Cleaning Toolkit

| Tool | What It Does | Example |
|---|---|---|
| `df.columns = [...]` | Replace all column names | `[c.lower() for c in df.columns]` |
| `.rename(columns={...})` | Rename specific columns | `df.rename(columns={"a": "b"})` |
| `.isna()` / `.notna()` | `True` where a value is missing / present | `df.isna().sum()` |
| `.dropna(subset=[...])` | Drop rows missing a value in those columns | `df.dropna(subset=["review_text"])` |
| `.fillna(value)` | Replace missing values | `df["title"].fillna("")` |
| `.duplicated(subset=[...])` | `True` for repeats (`keep=False` marks all copies) | `df.duplicated(subset=["review_text"])` |
| `.drop_duplicates(subset=[...])` | Keep only the first of each repeat | `df.drop_duplicates(subset=["review_text"])` |
| `.replace({old: new})` | Swap specific values | `.replace({"Initmates": "Intimates"})` |
| `.astype(type)` | Change a column's type | `df["recommended"].astype(bool)` |
| `pd.to_datetime(col)` | Convert text (or Unix seconds) to dates | `pd.to_datetime(df["date"])` |
| `.dt.year`, `.dt.month`, `.dt.day_name()` | Parts of a date | `df["date"].dt.year` |
| `.str.lower()`, `.str.strip()`, `.str.len()` | String methods on every value | `df["title"].str.lower()` |
| `.str.split().str.len()` | Word count per value | `df["review_text"].str.split().str.len()` |
| `.str.contains(p, case=False)` | Does each value match the pattern? | `.str.contains(r"\bpetite\b")` |
| `.str.count(p)` | Number of matches per value | `.str.count("!")` |
| `.str.replace(p, new, regex=True)` | Regex substitution per value | `.str.replace(r"\s+", " ", regex=True)` |
| `.str.extract(p)` | One column per regex group | `.str.extract(r"(\d)'(\d+)")` |
| `.apply(function)` | Run your own function on every value | `df["review_text"].apply(clean_text)` |
| `.map(dict)` | Translate values through a lookup table | `df["rating"].map(labels)` |
| `.to_csv(path, index=False)` | Save a DataFrame to a CSV file | `df.to_csv("clean.csv", index=False)` |

The whole lab in one sentence: **cleaning is a series of decisions (drop, fill, fix, or leave), each justified and checked with before-and-after counts, and the `.str` accessor and `.apply()` turn raw review text into features you can analyse.**